In [1]:
import cv2

# Initialize the webcam (0 is usually the default camera)
cap = cv2.VideoCapture(0)

# Check if the webcam is opened correctly
if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

# Capture a single frame
ret, frame = cap.read()

# If frame is captured successfully
if ret:
    # Display the frame in a window
    cv2.imshow('Captured Frame', frame)
    
    # Wait for a key press to close the window
    cv2.waitKey(0)

# Release the webcam and close the window
cap.release()
cv2.destroyAllWindows()

In [2]:
import cv2

# Initialize the webcam (0 is usually the default camera)
cap = cv2.VideoCapture(0)

# Check if the webcam is opened correctly
if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

# Loop to continuously capture frames
while True:
    # Capture each frame
    ret, frame = cap.read()

    # If the frame is captured successfully
    if ret:
        # Display the frame in a window
        cv2.imshow('Webcam Feed', frame)

        # Wait for 1 ms and check if 'q' is pressed to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    else:
        print("Error: Failed to capture frame.")
        break

# Release the webcam and close all OpenCV windows
cap.release()
cv2.destroyAllWindows()

In [3]:
cap.release()
cv2.destroyAllWindows()

In [5]:
import cv2
import numpy as np

# Open the default webcam. Fall back to the standard backend if DirectShow fails.
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap.release()
    cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError(
        "Could not open the webcam. Close other camera apps and try again."
    )

# HSV range for a blue marker.
LOWER_COLOR = np.array([100, 150, 0])
UPPER_COLOR = np.array([140, 255, 255])

drawing_mask = None
prev_point = None

print("Air Canvas started. Show a BLUE marker to draw.")
print("Press 'c' to clear the drawing or 'q' to quit.")

try:
    while True:
        success, frame = cap.read()
        if not success:
            print("Could not read a frame from the webcam.")
            break

        frame = cv2.flip(frame, 1)
        if drawing_mask is None:
            drawing_mask = np.zeros(frame.shape[:2], dtype=np.uint8)

        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        marker_mask = cv2.inRange(hsv, LOWER_COLOR, UPPER_COLOR)
        marker_mask = cv2.erode(marker_mask, None, iterations=1)
        marker_mask = cv2.dilate(marker_mask, None, iterations=1)

        contours, _ = cv2.findContours(
            marker_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        curr_point = None
        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            if cv2.contourArea(largest_contour) > 500:
                moments = cv2.moments(largest_contour)
                if moments['m00'] != 0:
                    curr_point = (
                        int(moments['m10'] / moments['m00']),
                        int(moments['m01'] / moments['m00']),
                    )
                    cv2.circle(frame, curr_point, 8, (0, 255, 255), -1)

        # Store strokes in a mask so black lines remain visible on the live video.
        if curr_point is not None and prev_point is not None:
            cv2.line(drawing_mask, prev_point, curr_point, 255, 5)
        prev_point = curr_point

        combined = frame.copy()
        combined[drawing_mask > 0] = (0, 0, 0)
        cv2.putText(
            combined,
            "Press 'c' to Clear | Press 'q' to Quit",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2,
        )
        cv2.imshow('BCA AI Demo - Virtual Air Canvas', combined)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('c'):
            drawing_mask.fill(0)
            prev_point = None
        elif key == ord('q'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()

Air Canvas started. Show a BLUE marker to draw.
Press 'c' to clear the drawing or 'q' to quit.
